This notebook prepares EB samples for geoparsing tasks.

In [1]:
import os

from SPARQLWrapper import SPARQLWrapper, JSON
# get short high quality articles text from 7th edition EB

sparql = SPARQLWrapper(
    "http://query.frances-ai.com/hto"
)
sparql.setReturnFormat(JSON)

In [7]:
def get_short_articles(min_length, max_length):
    articles = []
    sparql.setQuery("""
    PREFIX hto: <https://w3id.org/hto#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?term_uri ?name ?hq_desc ?hq_text WHERE {
        ?term_uri a hto:ArticleTermRecord;
            hto:name ?name;
            hto:startsAtPage ?startPage;
            hto:hasOriginalDescription ?hq_desc.
        ?hq_desc hto:text ?hq_text;
            hto:hasTextQuality hto:High.
  		# Calculate word count for high-quality description
    	BIND (STRLEN(REPLACE(?hq_text, "\\\\S", " ")) AS ?high_length)
    	FILTER (?high_length >= %s && ?high_length <= %s)
        ?vol a hto:Volume;
            hto:hadMember ?startPage.
        ?edition a hto:Edition;
            hto:hadMember ?vol;
            hto:yearPublished ?year_published.
        FILTER (?year_published = 1842)
        }
    """ % (min_length, max_length))

    try:
        ret = sparql.queryAndConvert()
        for r in ret["results"]["bindings"]:
            articles.append({
                "term_uri": r["term_uri"]["value"],
                "name": r["name"]["value"],
                "hq_text": r["hq_text"]["value"],
                "hq_desc_uri": r["hq_desc"]["value"]
            })
    except Exception as e:
        print(e)

    return articles

short_articles = get_short_articles(0, 1000)
len(short_articles)

17270

In [8]:
short_articles[0]

{'term_uri': 'https://w3id.org/hto/ArticleTermRecord/9910796273804340_192693199_1002446047_0',
 'name': 'IMPEDIMENTS',
 'hq_text': 'in Law, are such hindrances as put a stop or stay to a person’s seeking his right by due course of law. Persons under impediments are those under age or coverture, non compos mentis, in prison, beyond sea, and the like, who, by a saving in our laws, have time to claim and prosecute their rights, after the impediments are removed.',
 'hq_desc_uri': 'https://w3id.org/hto/OriginalDescription/9910796273804340_192693199_1002446047_0NCKP'}

In [9]:
import pandas as pd
eb_geo_samples = pd.DataFrame(short_articles)
eb_geo_samples

,term_uri,name,hq_text,hq_desc_uri
0,https://w3id.org/hto/ArticleTermRecord/9910796...,IMPEDIMENTS,"in Law, are such hindrances as put a stop or s...",https://w3id.org/hto/OriginalDescription/99107...
1,https://w3id.org/hto/ArticleTermRecord/9910796...,KASIMOW,"a city of the Russian province Riasan, the cap...",https://w3id.org/hto/OriginalDescription/99107...
2,https://w3id.org/hto/ArticleTermRecord/9910796...,INTRASCA,"a city of the kingdom of Sardinia, in the prov...",https://w3id.org/hto/OriginalDescription/99107...
3,https://w3id.org/hto/ArticleTermRecord/9910796...,ILLUMINATI,the name of a secret society or order in Germa...,https://w3id.org/hto/OriginalDescription/99107...
4,https://w3id.org/hto/ArticleTermRecord/9910796...,KAUFBEUERN,"a city of Bavaria, in the province of the Uppe...",https://w3id.org/hto/OriginalDescription/99107...
...,...,...,...,...
17265,https://w3id.org/hto/ArticleTermRecord/9910796...,NYKOPING,"a province of the west of Sweden, which extend...",https://w3id.org/hto/OriginalDescription/99107...
17266,https://w3id.org/hto/ArticleTermRecord/9910796...,PALFREY,is one of the better sort of horses used by no...,https://w3id.org/hto/OriginalDescription/99107...
17267,https://w3id.org/hto/ArticleTermRecord/9910796...,NEWCASTLE,"a small market-town of South Wales, in the cou...",https://w3id.org/hto/OriginalDescription/99107...
17268,https://w3id.org/hto/ArticleTermRecord/9910796...,NIESTER,"a large river of Polish Russia, having its sou...",https://w3id.org/hto/OriginalDescription/99107...


In [10]:
# save this samples
eb_geo_samples.to_json("eb_geo_samples.json", orient="records", lines=True)

## Prepare all EB 7th edition articles for geoparsing

In [6]:
def get_articles(year):
    articles = []
    sparql.setQuery("""
    PREFIX hto: <https://w3id.org/hto#>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?term_uri ?name ?hq_desc ?hq_text WHERE {
        ?term_uri a ?term_type;
            hto:name ?name;
            hto:startsAtPage ?startPage;
            hto:hasOriginalDescription ?hq_desc.
        FILTER (?term_type = hto:ArticleTermRecord || ?term_type = hto:TopicTermRecord)
        ?hq_desc hto:text ?hq_text;
            hto:hasTextQuality hto:High.
  		# Calculate word count for high-quality description
    	BIND (STRLEN(REPLACE(?hq_text, "\\\\S", " ")) AS ?high_length)
    	FILTER (?high_length > 0)
        ?vol a hto:Volume;
            hto:hadMember ?startPage.
        ?edition a hto:Edition;
            hto:hadMember ?vol;
            hto:yearPublished ?year_published.
        FILTER (?year_published = %s)
        }
    """ % year)

    try:
        ret = sparql.queryAndConvert()
        for r in ret["results"]["bindings"]:
            articles.append({
                "term_uri": r["term_uri"]["value"],
                "name": r["name"]["value"],
                "hq_text": r["hq_text"]["value"],
                "hq_desc_uri": r["hq_desc"]["value"]
            })
    except Exception as e:
        print(e)

    return articles

e7_articles = get_articles(1842)
len(e7_articles)

23965

In [8]:
import pandas as pd
e7_geo = pd.DataFrame(e7_articles)
e7_geo.head()

,term_uri,name,hq_text,hq_desc_uri
0,https://w3id.org/hto/ArticleTermRecord/9910796...,IMPEDIMENTS,"in Law, are such hindrances as put a stop or s...",https://w3id.org/hto/OriginalDescription/99107...
1,https://w3id.org/hto/ArticleTermRecord/9910796...,KASIMOW,"a city of the Russian province Riasan, the cap...",https://w3id.org/hto/OriginalDescription/99107...
2,https://w3id.org/hto/ArticleTermRecord/9910796...,INTRASCA,"a city of the kingdom of Sardinia, in the prov...",https://w3id.org/hto/OriginalDescription/99107...
3,https://w3id.org/hto/ArticleTermRecord/9910796...,ILLUMINATI,the name of a secret society or order in Germa...,https://w3id.org/hto/OriginalDescription/99107...
4,https://w3id.org/hto/ArticleTermRecord/9910796...,KAUFBEUERN,"a city of Bavaria, in the province of the Uppe...",https://w3id.org/hto/OriginalDescription/99107...


In [9]:
e7_geo.to_json("eb7th_geo_input.json", orient="records", lines=True)

## Prepare all EB 1st edition 1771 articles for geoparsing

In [7]:
e1_1771_articles = get_articles(1771)
len(e1_1771_articles)

15931

In [8]:
import pandas as pd
e1_1771_geo = pd.DataFrame(e1_1771_articles)

AttributeError: 'list' object has no attribute 'head'

In [9]:
e1_1771_geo.head()

,term_uri,name,hq_text,hq_desc_uri
0,https://w3id.org/hto/ArticleTermRecord/9922776...,ALIEMBUT,"in botany, an obsolete name of a species of mi...",https://w3id.org/hto/OriginalDescription/99227...
1,https://w3id.org/hto/ArticleTermRecord/9922776...,APERTURE,"the opening of any thing, or a hole or cleft i...",https://w3id.org/hto/OriginalDescription/99227...
2,https://w3id.org/hto/ArticleTermRecord/9922776...,BLOCK,"a large mals of wood, serving to work or cut t...",https://w3id.org/hto/OriginalDescription/99227...
3,https://w3id.org/hto/ArticleTermRecord/9922776...,ASTROSCOPE,"an instrument composed of two canes, raving th...",https://w3id.org/hto/OriginalDescription/99227...
4,https://w3id.org/hto/ArticleTermRecord/9922776...,ALZIRA,"a town of Spain, in the province of Valentia, ...",https://w3id.org/hto/OriginalDescription/99227...


In [10]:
e1_1771_geo.to_json("eb1_1771_geo_input.json", orient="records", lines=True)